# 01 — Ingestion quotidienne des taux de change BCE

Exécuté par le Job `agrimeca_fx_daily`, du lundi au vendredi à 17 h (heure de Paris),
après la publication des taux de référence BCE (vers 16 h).

- réutilise `scripts/ingest_ecb.py` du dépôt (mêmes fonctions qu'en local) ;
- récupère les taux des `days_back` derniers jours : le chevauchement couvre les jours fériés
  et un éventuel run manqué, le Silver garde un seul taux par devise et par jour ;
- écrit la réponse brute dans `/Volumes/supplychain/landing/erp/fx/ingest_date=…/run_id=…/exr.csv`.

In [ ]:
# Cellule 1 - paramètres (modifiables dans le Job sans toucher au code)
dbutils.widgets.text("landing", "/Volumes/supplychain/landing/erp")
dbutils.widgets.text("days_back", "10")

LANDING = dbutils.widgets.get("landing")
DAYS_BACK = int(dbutils.widgets.get("days_back"))

In [ ]:
# Cellule 2 - importer le script du dépôt (dossier scripts/, à côté de Notebook/)
import os
import sys

sys.path.append(os.path.abspath("../scripts"))
from ingest_ecb import CURRENCIES, fetch_rates, save_rates  # noqa: E402

In [ ]:
# Cellule 3 - appel de l'API BCE et dépôt dans le Volume
from datetime import date, timedelta

start = (date.today() - timedelta(days=DAYS_BACK)).isoformat()
csv_text = fetch_rates(start)
folder = save_rates(csv_text, LANDING)

n_rows = len(csv_text.splitlines()) - 1  # moins la ligne d'en-tête
print(f"fx : {n_rows} taux ({', '.join(CURRENCIES)}) depuis {start} -> {folder}")

In [ ]:
# Cellule 4 - contrôle : un run sans aucun taux fait échouer le Job (et déclenche l'alerte)
if n_rows < len(CURRENCIES):
    raise ValueError(f"seulement {n_rows} taux reçus depuis {start} : réponse BCE anormale")

dbutils.notebook.exit(f"{n_rows} taux depuis {start}")  # résumé visible dans l'historique du Job